In [ ]:
Obtain annual FVC maps for cropland in Jiangsu Province


In [1]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

clcd_path = Path('/Users/wangze/Dropbox/Emi/LandControl/geodata/CLCD_v01_2016_albert_province/CLCD_v01_2016_albert_jiangsu.tif')

In [2]:
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import glob
from pathlib import Path

# === 路径设置 ===
fvc_dir = Path("/Users/wangze/cropland_data/fvc/2016")
fvc_dev = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_fvc")

# 可选：定义要处理的 tile 列表
tiles = ["N50_30", "N51_30"]

# 目标投影系 (China CGCS2000 / Gauss–Kruger Zone 19)
TARGET_CRS = "EPSG:4547"

for tile in tiles:
    print(f"\n🌍 Processing tile: {tile} ...")

    # === Step 1. 找到该 tile 的 24 期文件 ===
    files = sorted(glob.glob(str(fvc_dir / f"{tile}_FVC-2016-*-30m.tif")))
    if len(files) == 0:
        print(f"⚠️ No files found for {tile}")
        continue
    print(f"  Found {len(files)} half-monthly files.")

    # === Step 2. 打开第一个文件，建立基准 ===
    with rasterio.open(files[0]) as ref:
        ref_meta = ref.meta.copy()
        ref_shape = (ref.height, ref.width)
        ref_crs = ref.crs
        ref_transform = ref.transform

    # === Step 3. 初始化累积矩阵 ===
    mean_fvc = np.zeros(ref_shape, dtype=np.float64)
    count = np.zeros(ref_shape, dtype=np.int32)

    # === Step 4. 循环累加 ===
    for t, f in enumerate(files, 1):
        with rasterio.open(f) as src:
            arr = src.read(1).astype(np.float32)
            arr[arr > 100] = np.nan  # 无效值
            arr /= 100.0
            mask = np.isfinite(arr)
            mean_fvc[mask] += arr[mask]
            count[mask] += 1
        print(f"  ✅ processed {t:02d}/24")

    # === Step 5. 计算平均值 ===
    mean_fvc = np.divide(mean_fvc, count, out=np.zeros_like(mean_fvc), where=count > 0)
    mean_fvc = mean_fvc.astype(np.float32)

    # === Step 6. 保存为原始坐标系临时文件 ===
    temp_path = fvc_dev / f"FVC_2016_{tile}_mean_raw.tif"
    out_meta = ref_meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "dtype": "float32",
        "count": 1
    })
    with rasterio.open(temp_path, "w", **out_meta) as dst:
        dst.write(mean_fvc, 1)
    print(f"  💾 Saved temporary mean file → {temp_path}")

    # === Step 7. 重投影为 EPSG:4547 ===
    reprojected_path = fvc_dev / f"FVC_2016_{tile}_mean_4547.tif"
    with rasterio.open(temp_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, TARGET_CRS, src.width, src.height, *src.bounds
        )
        kwargs = src.meta.copy()
        kwargs.update({
            "crs": TARGET_CRS,
            "transform": transform,
            "width": width,
            "height": height,
            "dtype": "float32"
        })

        with rasterio.open(reprojected_path, "w", **kwargs) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=TARGET_CRS,
                resampling=Resampling.bilinear
            )
    print(f"  🌐 Reprojected to EPSG:4547 → {reprojected_path}")

print("\n✅ All tiles processed & reprojected successfully!")


🌍 Processing tile: N50_30 ...
  Found 24 half-monthly files.
  ✅ processed 01/24
  ✅ processed 02/24
  ✅ processed 03/24
  ✅ processed 04/24
  ✅ processed 05/24
  ✅ processed 06/24
  ✅ processed 07/24
  ✅ processed 08/24
  ✅ processed 09/24
  ✅ processed 10/24
  ✅ processed 11/24
  ✅ processed 12/24
  ✅ processed 13/24
  ✅ processed 14/24
  ✅ processed 15/24
  ✅ processed 16/24
  ✅ processed 17/24
  ✅ processed 18/24
  ✅ processed 19/24
  ✅ processed 20/24
  ✅ processed 21/24
  ✅ processed 22/24
  ✅ processed 23/24
  ✅ processed 24/24
  💾 Saved temporary mean file → /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_fvc/FVC_2016_N50_30_mean_raw.tif
  🌐 Reprojected to EPSG:4547 → /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_fvc/FVC_2016_N50_30_mean_4547.tif

🌍 Processing tile: N51_30 ...
  Found 24 half-monthly files.
  ✅ processed 01/24
  ✅ processed 02/24
  ✅ processed 03/24
  ✅ processed 04/24
  ✅ processed 05/24
  ✅ processed 06/24
  ✅ processed 07/24
  ✅ processed 08/2

In [3]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from pathlib import Path

# 输入和输出路径
src_path = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/CLCD_v01_2016_albert_province/CLCD_v01_2016_albert_jiangsu.tif")
dst_path = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/CLCD_v01_2016_albert_jiangsu_albers.tif")

# 目标投影（与 FVC 一致）
dst_crs = "EPSG:4547"   # CGCS2000 / Albers Equal Area Conic

with rasterio.open(src_path) as src:
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )
    kwargs = src.meta.copy()
    kwargs.update({
        'crs': dst_crs,
        'transform': transform,
        'width': width,
        'height': height
    })

    with rasterio.open(dst_path, 'w', **kwargs) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest  # 分类数据必须用 nearest
            )

print("✅ 重投影完成！已生成文件：", dst_path)



✅ 重投影完成！已生成文件： /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/CLCD_v01_2016_albert_jiangsu_albers.tif


In [4]:
import rasterio
from rasterio.merge import merge
from rasterio.enums import Resampling
from pathlib import Path

base_dir = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata")
fvc50_path = base_dir / "Jiangsu_fvc/FVC_2016_N50_30_mean_4547.tif"
fvc51_path = base_dir / "Jiangsu_fvc/FVC_2016_N51_30_mean_4547.tif"

# 打开两个文件
srcs = [rasterio.open(fvc50_path), rasterio.open(fvc51_path)]

# 手动计算并扩大 bounds
all_bounds = [s.bounds for s in srcs]
minx = min(b.left for b in all_bounds)
maxx = max(b.right for b in all_bounds)
miny = min(b.bottom for b in all_bounds)
maxy = max(b.top for b in all_bounds)
full_bounds = (minx, miny, maxx, maxy)

# 拼接（保留全部范围）
mosaic, out_trans = merge(
    srcs,
    bounds=full_bounds,
    res=srcs[0].res,  # 保持相同分辨率
    method="max",     # 优先保留较大像元（即非空值）
    precision=1       # 允许轻微格网容差
)

# 保存
out_meta = srcs[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans,
    "crs": "EPSG:4547",
    "dtype": "float32",
    "count": 1
})

out_path = base_dir / "Jiangsu_fvc/FVC_2016_Jiangsu_merged_full_4547.tif"
with rasterio.open(out_path, "w", **out_meta) as dst:
    dst.write(mosaic[0], 1)

print(f"✅ Saved full merged map → {out_path}")

/Users/wangze/Dropbox/Emi/LandControl/.venv/lib/python3.12/site-packages/rasterio/merge.py:217: RasterioDeprecationWarning: The precision parameter is unused, deprecated, and will be removed in 2.0.0.
  warnings.warn(


✅ Saved full merged map → /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_fvc/FVC_2016_Jiangsu_merged_full_4547.tif


In [7]:
# ===== Jiangsu cropland fallow mapping (CLCD × merged FVC) =====
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- paths ----------
base = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata")
clcd_path  = base / "Jiangsu_CGCS2000/CLCD_v01_2016_albert_jiangsu_albers.tif"
fvc_path   = base / "Jiangsu_fvc/FVC_2016_Jiangsu_merged_full_4547.tif"  # 上一步合并好的那个
out_dir    = base / "Jiangsu_fvc"
out_dir.mkdir(parents=True, exist_ok=True)

# ---------- params ----------
CROPLAND_CODES = {1}        # 你已确认 CLCD 耕地=1；如有子类可写成 {11,12}
FALLOW_THR = 0.30           # FVC 低于此阈值视为撂荒（可改成 0.25 或 0.35 做敏感性分析）
FALLOW_NODATA = 255         # 输出栅格 nodata

# ---------- read rasters ----------
with rasterio.open(clcd_path) as clcd_src:
    clcd = clcd_src.read(1)
    clcd_meta = clcd_src.meta.copy()
    clcd_crs  = clcd_src.crs
    clcd_tf   = clcd_src.transform
    clcd_shape = (clcd_src.height, clcd_src.width)

with rasterio.open(fvc_path) as fvc_src:
    fvc = fvc_src.read(1).astype(np.float32)
    fvc_crs = fvc_src.crs
    fvc_tf  = fvc_src.transform

print("CRS:", "\n  CLCD:", clcd_crs, "\n  FVC :", fvc_crs)

# ---------- align FVC to CLCD grid (only if needed) ----------
if fvc_crs != clcd_crs or fvc.shape != clcd_shape or fvc_tf != clcd_tf:
    aligned_fvc = np.full(clcd_shape, np.nan, dtype=np.float32)
    reproject(
        source=fvc,
        destination=aligned_fvc,
        src_transform=fvc_tf,
        src_crs=fvc_crs,
        dst_transform=clcd_tf,
        dst_crs=clcd_crs,
        resampling=Resampling.bilinear,
        src_nodata=None,
        dst_nodata=np.nan,
    )
    fvc = aligned_fvc
    print("✅ FVC 已对齐到 CLCD 网格。")
else:
    print("✅ FVC 与 CLCD 天然对齐。")

# ---------- sanitize FVC ----------
# 有些工具写出的 FVC 可能有轻微越界或 0/负值；统一限定在 [0,1]，其余设为 NaN
fvc = fvc.astype(np.float32)
fvc[~np.isfinite(fvc)] = np.nan
fvc[(fvc < 0) | (fvc > 1)] = np.nan

# ---------- cropland mask ----------
if len(CROPLAND_CODES) == 1:
    cropland_mask = (clcd == next(iter(CROPLAND_CODES)))
else:
    # 多码写法：如 {11,12}
    cropland_mask = np.isin(clcd, list(CROPLAND_CODES))

valid_on_cropland = cropland_mask & np.isfinite(fvc)

# ---------- fallow map (0/1 with nodata) ----------
fallow_bool = np.zeros_like(fvc, dtype=bool)
fallow_bool[valid_on_cropland] = (fvc[valid_on_cropland] < FALLOW_THR)

fallow_u8 = np.full(fvc.shape, FALLOW_NODATA, dtype=np.uint8)  # 255=nodata
fallow_u8[valid_on_cropland] = fallow_bool[valid_on_cropland].astype(np.uint8)

# ---------- stats ----------
total_cropland = int(cropland_mask.sum())
valid_cropland = int(valid_on_cropland.sum())
fallow_pixels  = int(fallow_bool.sum())
fallow_rate    = (fallow_pixels / valid_cropland) if valid_cropland > 0 else np.nan

print("\n📊 撂荒统计（基于 FVC < {:.2f}）：".format(FALLOW_THR))
print("  耕地像元总数：{:,.0f}".format(total_cropland))
print("  有效FVC耕地像元：{:,.0f}".format(valid_cropland))
print("  撂荒像元数：{:,.0f}".format(fallow_pixels))
print("  撂荒率：{:.2%}".format(fallow_rate))

# === 生成耕地区域内的 FVC 连续图 ===
fvc_cropland = np.full_like(fvc, np.nan, dtype=np.float32)
fvc_cropland[valid_on_cropland] = fvc[valid_on_cropland]

out_fvc_tif = out_dir / "FVC_Jiangsu_2016_cropland_only_4547.tif"

fvc_meta = clcd_meta.copy()
fvc_meta.update({
    "driver": "GTiff",
    "dtype": "float32",
    "count": 1,
    "nodata": np.nan
})

with rasterio.open(out_fvc_tif, "w", **fvc_meta) as dst:
    dst.write(fvc_cropland, 1)

print(f"✅ 连续型 FVC（仅耕地）已保存至: {out_fvc_tif}")

CRS: 
  CLCD: EPSG:4547 
  FVC : EPSG:4547
✅ FVC 已对齐到 CLCD 网格。

📊 撂荒统计（基于 FVC < 0.30）：
  耕地像元总数：78,732,414
  有效FVC耕地像元：78,732,414
  撂荒像元数：11,789,444
  撂荒率：14.97%
✅ 连续型 FVC（仅耕地）已保存至: /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_fvc/FVC_Jiangsu_2016_cropland_only_4547.tif
